# Research: Earnings Volatility Ramp

## Objective
Test the hypothesis that straddles held through an earnings event are not sufficient for profitability.

## Step 1: Universe Selection
Obtain all tickers whose daily option volume > 20,000 using dynamic universe selection.
High liquid options are required for this strategy.

In [ ]:
from AlgorithmImports import *
import pandas as pd

# Initialize QuantBook
qb = QuantBook()

# Set Analysis Date (e.g., peak of earnings season or a recent date)
selected_date = datetime(2023, 10, 25)
qb.SetStartDate(selected_date)
qb.SetEndDate(selected_date)

# -----------------------------------------------------
# Define Tickers via Universe Selection
# -----------------------------------------------------
# Instead of hardcoding, we use Coarse Fundamental data to find the most liquid stocks.
# This acts as a proxy for option liquidity.

print(f"Fetching Universe for {selected_date.date()}...")

# qb.GetFundamental returns a list of CoarseFundamental objects for the given date
coarse_data = qb.GetFundamental(selected_date)

# Filter constraints
min_price = 20
top_count = 50 # Limit to top 50 to keep processing time reasonable in a notebook

# Filter: Price > 20 and HasFundamentalData
filtered_coarse = [x for x in coarse_data if x.Price > min_price and x.HasFundamentalData]

# Sort by DollarVolume (Liquidity) and take top N
top_liquid = sorted(filtered_coarse, key=lambda x: x.DollarVolume, reverse=True)[:top_count]

# Extract ticker strings
tickers = [x.Symbol.Value for x in top_liquid]

option_vol_threshold = 20000
valid_tickers = []

print(f"Selected Top {len(tickers)} Tickers by Dollar Volume. Scanning for Option Volume > {option_vol_threshold}...")

for ticker in tickers:
    try:
        # Add Equity and Option
        # We use Minute resolution to capture granular volume if needed, 
        # but for volume sum, we can iterate the returned history.
        equity = qb.AddEquity(ticker, Resolution.Minute)
        option = qb.AddOption(ticker, Resolution.Minute)
        
        # Set Filter: Broad enough to capture the bulk of volume (ATM +/- 10 strikes, near expiry)
        # Volume typically concentrates in near-term ATM/OTM options.
        option.SetFilter(lambda u: u.IncludeWeeklys().Strikes(-20, 20).Expiration(0, 60))
        
        # Request Option History for the single day
        # GetOptionHistory returns an iterable of Slices containing OptionChains
        history = qb.GetOptionHistory(equity.Symbol, selected_date, selected_date)
        
        daily_volume = 0
        
        # Iterate through all time slices (minutes) in the day
        for slices in history:
            # Check if we have data for our symbol
            if option.Symbol in slices.OptionChains:
                chain = slices.OptionChains[option.Symbol]
                for contract in chain:
                    # Sum up volume for all contracts in the chain this minute
                    daily_volume += contract.Volume
        
        if daily_volume > option_vol_threshold:
            valid_tickers.append({'Ticker': ticker, 'Volume': daily_volume})
            print(f"MATCH: {ticker} | Volume: {daily_volume}")
            
    except Exception as e:
        # print(f"Skipping {ticker}: {e}")
        pass

# Display Results
df_result = pd.DataFrame(valid_tickers).sort_values(by='Volume', ascending=False)
print("\nSelected Tickers:")
print(df_result)

# Plotting (Optional)
if not df_result.empty:
    df_result.set_index('Ticker')['Volume'].plot(kind='bar', figsize=(15, 6), title='Option Volume by Ticker')
